# 04 · Base vs adapter, on the same rows

Set `VLM_ADAPTER_REPOSITORY` in `backend/.env` to the repository notebook 03 pushed, restart the kernel,
and run. The table joins this run's scores to `scores_baseline.csv` from notebook 02.

**What counts as success.** Not "higher on average". Three specific things:

1. **Bench rows** (human-verified) improve, not only the template `test` rows - a model can learn the
   template generator's habits without learning the ground.
2. **RSVQA-LR** (a different country, a different question generator) improves or holds. If BigEarthNet
   rows improved and RSVQA fell, the adapter over-fitted one dataset's phrasing; the fix is a lower rank
   or fewer BigEarthNet rows, not more epochs.
3. **Confidence tracks accuracy** more closely than before.

**The paired test.** For every row both models answered, count the rows the adapter fixed (base wrong,
adapter right) against the rows it broke. McNemar's exact test gives the probability of a split that
lopsided if the two were equally good; below 0.05 the difference is not noise. This is the number to
quote, not the difference of two means.

**If a type got worse** - captions often do at first, because ROUGE-L rewards the template's wording and
the base model's fluent generic caption scores surprisingly well on it - read the examples before
deciding; a caption that names the right land cover in different words is not a regression.

In [ ]:
import sys, pathlib, asyncio, json
BACKEND = pathlib.Path.cwd().resolve()
while BACKEND.name != "backend":
    BACKEND = BACKEND.parent
sys.path.insert(0, str(BACKEND))
import os; os.chdir(BACKEND)
import pandas as pd
from app.config import settings
from app.models.manager import get_manager, reset_manager
from app.services.evaluation.vqa import evaluate_vqa

# The same three files and limits for the base and for the adapter, so rows pair up exactly.
FILES = {
    "bigearthnet_txt.bench": (BACKEND / "data/training/vlm/bigearthnet_txt.bench.jsonl", None),   # 189 human-verified
    "bigearthnet_txt.test": (BACKEND / "data/training/vlm/bigearthnet_txt.test.jsonl", 400),     # template rows
    "rsvqa_lr.test": (BACKEND / "data/training/vlm/rsvqa_lr.test.jsonl", 550),                    # another country
}
PREDICTIONS = BACKEND / "data/training/vlm/predictions"

async def score(tag):
    manager = await get_manager()
    reports = {}
    for name, (file, limit) in FILES.items():
        if not file.exists():
            print("missing", file); continue
        reports[name] = await evaluate_vqa(manager=manager, file=file, limit=limit,
                                           predictions_path=PREDICTIONS / f"{tag}.{name}.jsonl")
    await reset_manager()
    return reports

def table(reports):
    rows = []
    for name, report in reports.items():
        for kind in sorted(report.score.asked):
            rows.append({"file": name, "type": kind, "n": report.score.asked[kind],
                         "score": round(report.score.accuracy(kind), 3), "model": report.model_version})
        rows.append({"file": name, "type": "OVERALL (mean of types)", "n": report.samples,
                     "score": round(report.score.overall, 3), "model": report.model_version})
    return pd.DataFrame(rows)

In [ ]:
assert settings.vlm_adapter_repository, "set VLM_ADAPTER_REPOSITORY to the pushed adapter"
reports = await score("adapter")
adapted = table(reports)
adapted.to_csv(BACKEND / "data/training/vlm/scores_adapter.csv", index=False)
adapted

In [ ]:
from training.vlm.compare import compare
frames = []
for name in FILES:
    base_file, adapter_file = PREDICTIONS / f"base.{name}.jsonl", PREDICTIONS / f"adapter.{name}.jsonl"
    if base_file.exists() and adapter_file.exists():
        frame = pd.DataFrame(compare(base_file, adapter_file)); frame.insert(0, "file", name); frames.append(frame)
paired = pd.concat(frames, ignore_index=True)
paired["delta"] = (paired["adapter"] - paired["base"]).round(3)
paired.round(3)

In [ ]:
for name, report in reports.items():
    print(f"== {name}  mean stated confidence {report.mean_confidence:.2f}")
    for kind, prompt, reference, prediction, hit in report.examples[:6]:
        print(f"  [{'ok ' if hit else 'BAD'}] {kind:<13} Q: {prompt[:80]}")
        print(f"                     ref: {reference[:60]!r}   got: {prediction[:60]!r}")